# Real-Time UPI Fraud Detection System: Folder Structure

This notebook creates a clean, production-friendly folder structure for a machine learning project that simulates UPI transactions and detects anomalous transactions with unsupervised anomaly detection.

Reference used: `UPI_Fraud_Detection.pdf`, which defines the synthetic transaction schema, six behavioral features, Isolation Forest training flow, F1-based threshold tuning, and Streamlit real-time demo.

The scaffold supports:

- Synthetic UPI transaction generation
- Fraud-focused behavioral feature engineering
- Isolation Forest model training
- Threshold tuning with F1-score
- Real-time Streamlit dashboard deployment

## 1. Imports and Configuration

In [9]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from textwrap import dedent
from typing import Iterable


PROJECT_NAME = "Real-Time-UPI-Fraud-Detection-System"
PACKAGE_NAME = "rtufd"
PROJECT_ROOT = Path.cwd() / PROJECT_NAME

## 2. Project Layout Definition

In [10]:
@dataclass(frozen=True)
class ProjectLayout:
    """Container for project directories and scaffold files."""

    root: Path
    package_name: str

    @property
    def directories(self) -> tuple[Path, ...]:
        package_root = self.root / "src" / self.package_name
        return (
            self.root / "app",
            self.root / "config",
            self.root / "data" / "raw",
            self.root / "data" / "interim",
            self.root / "data" / "processed",
            self.root / "data" / "synthetic",
            self.root / "models",
            self.root / "notebooks",
            self.root / "reports",
            self.root / "reports" / "figures",
            package_root,
            self.root / "tests",
        )

    @property
    def files(self) -> dict[Path, str]:
        package_root = self.root / "src" / self.package_name
        return {
            self.root / "README.md": readme_template(),
            self.root / ".gitignore": gitignore_template(),
            self.root / "requirements.txt": requirements_template(),
            self.root / "config" / "config.yaml": config_template(),
            self.root / "app" / "streamlit_app.py": streamlit_template(),
            package_root / "__init__.py": package_init_template(),
            package_root / "config.py": config_module_template(),
            package_root / "data_generation.py": data_generation_template(),
            package_root / "features.py": features_template(),
            package_root / "modeling.py": modeling_template(),
            package_root / "inference.py": inference_template(),
            package_root / "utils.py": utils_template(),
            self.root / "tests" / "test_project_structure.py": tests_template(),
        }

## 3. Scaffold File Templates

In [11]:
def clean_text(value: str) -> str:
    """Normalize indentation and ensure each generated text file ends with one newline."""
    return dedent(value).strip() + "\n"


def readme_template() -> str:
    return clean_text(
        """
        # Real-Time UPI Fraud Detection System

        A machine learning-based anomaly detection system that learns the normal transaction fingerprint of each user and flags transactions that deviate significantly from expected behavior.

        ## Features

        - Generate synthetic UPI transaction data
        - Engineer user behavior and transaction-risk features
        - Train an Isolation Forest anomaly detection model
        - Tune anomaly thresholds with F1-score
        - Serve a real-time fraud detection dashboard with Streamlit

        ## Suggested Workflow

        1. Generate synthetic transactions into `data/synthetic/`.
        2. Build features and save processed datasets into `data/processed/`.
        3. Train and tune the model, then save artifacts into `models/`.
        4. Run the dashboard with `streamlit run app/streamlit_app.py`.
        """
    )


def gitignore_template() -> str:
    return clean_text(
        """
        __pycache__/
        .pytest_cache/
        .ipynb_checkpoints/
        .venv/
        venv/
        *.pyc
        *.pkl
        *.joblib
        data/raw/*
        data/interim/*
        data/processed/*
        data/synthetic/*
        models/*
        reports/figures/*
        !.gitkeep
        """
    )


def requirements_template() -> str:
    return clean_text(
        """
        numpy
        pandas
        scikit-learn
        joblib
        pyyaml
        streamlit
        matplotlib
        seaborn
        plotly
        pytest
        """
    )


def config_template() -> str:
    return clean_text(
        """
        project:
          name: Real-Time-UPI-Fraud-Detection-System
          random_state: 42

        data:
          synthetic_rows: 10000
          fraud_rate: 0.03

        model:
          contamination: 0.04
          n_estimators: 300
          threshold_candidates: 100
        """
    )

In [12]:
def package_init_template() -> str:
    return clean_text(
        '''
        """Utilities for Real-Time UPI Fraud Detection System."""

        __version__ = "0.1.0"
        '''
    )


def config_module_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        from pathlib import Path
        from typing import Any

        import yaml


        def load_config(config_path: str | Path) -> dict[str, Any]:
            """Load a YAML configuration file."""
            path = Path(config_path)
            with path.open("r", encoding="utf-8") as file:
                return yaml.safe_load(file)
        '''
    )


def data_generation_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        import numpy as np
        import pandas as pd


        def generate_synthetic_upi_transactions(
            rows: int = 10000,
            fraud_rate: float = 0.03,
            random_state: int = 42,
        ) -> pd.DataFrame:
            """Generate PDF-aligned synthetic UPI transactions with sparse fraud labels."""
            rng = np.random.default_rng(random_state)
            start_time = pd.Timestamp("2024-08-01 00:00:00")
            timestamps = start_time + pd.to_timedelta(rng.integers(0, 30 * 24 * 60, size=rows), unit="m")
            sender_ids = rng.integers(1000, 1800, size=rows)
            receiver_ids = rng.integers(2000, 3200, size=rows)
            is_fraud = rng.random(rows) < fraud_rate

            amounts = rng.lognormal(mean=5.4, sigma=0.8, size=rows)
            amounts[is_fraud] *= rng.uniform(4.0, 12.0, size=is_fraud.sum())
            amounts = amounts.round(2)

            timestamps = pd.Series(timestamps)
            timestamps.loc[is_fraud] = timestamps.loc[is_fraud].dt.normalize() + pd.to_timedelta(
                rng.choice([1, 2, 3, 4], size=is_fraud.sum()), unit="h"
            )
            receiver_ids[is_fraud] = rng.integers(9000, 9999, size=is_fraud.sum())

            return pd.DataFrame(
                {
                    "transaction_id": [f"TXN_202408_{i:05d}" for i in range(rows)],
                    "timestamp": timestamps,
                    "sender_account_id": [f"ACC_{account_id}" for account_id in sender_ids],
                    "receiver_account_id": [f"ACC_{account_id}" for account_id in receiver_ids],
                    "amount": amounts,
                    "location_pincode": rng.choice(["400001", "560001", "110001", "700001", "600001"], size=rows),
                    "transaction_type": rng.choice(["P2P", "Merchant"], size=rows, p=[0.72, 0.28]),
                    "is_fraud": is_fraud.astype(int),
                }
            )
        '''
    )


def features_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        import pandas as pd


        FEATURE_COLUMNS = [
            "hour_of_day",
            "day_of_week",
            "transaction_value",
            "is_first_time_receiver",
            "velocity_last_10min",
            "amount_vs_user_avg",
        ]


        def build_behavioral_features(transactions: pd.DataFrame) -> pd.DataFrame:
            """Create the six behavioral features highlighted in the project guide."""
            features = transactions.copy()
            features["timestamp"] = pd.to_datetime(features["timestamp"])
            features = features.sort_values(["sender_account_id", "timestamp"]).reset_index(drop=True)

            features["hour_of_day"] = features["timestamp"].dt.hour
            features["day_of_week"] = features["timestamp"].dt.dayofweek
            features["transaction_value"] = features["amount"]
            features["is_first_time_receiver"] = (~features.duplicated(["sender_account_id", "receiver_account_id"])).astype(int)
            features["velocity_last_10min"] = calculate_velocity(features)

            user_average = features.groupby("sender_account_id")["amount"].transform("mean")
            features["amount_vs_user_avg"] = features["amount"] / user_average.clip(lower=1)
            return features


        def calculate_velocity(transactions: pd.DataFrame) -> pd.Series:
            """Count each sender's transactions in a rolling 10-minute window."""
            velocity = pd.Series(index=transactions.index, dtype=float)

            for _, group in transactions.groupby("sender_account_id"):
                ordered = group.sort_values("timestamp").set_index("timestamp")
                counts = ordered["transaction_id"].rolling("10min").count()
                velocity.loc[group.sort_values("timestamp").index] = counts.to_numpy()

            return velocity.fillna(1)
        '''
    )

In [13]:
def modeling_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        import numpy as np
        import pandas as pd
        from sklearn.ensemble import IsolationForest
        from sklearn.metrics import f1_score
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler


        def train_isolation_forest(
            features: pd.DataFrame,
            contamination: float = 0.04,
            n_estimators: int = 300,
            random_state: int = 42,
        ) -> Pipeline:
            """Train an Isolation Forest pipeline on behavioral features."""
            return Pipeline(
                steps=[
                    ("scaler", StandardScaler()),
                    (
                        "model",
                        IsolationForest(
                            contamination=contamination,
                            n_estimators=n_estimators,
                            random_state=random_state,
                        ),
                    ),
                ]
            ).fit(features)


        def tune_threshold(scores: np.ndarray, labels: pd.Series, candidates: int = 100) -> tuple[float, float]:
            """Select the anomaly-score threshold with the best F1-score."""
            best_threshold = float(scores.min())
            best_f1 = 0.0

            for threshold in np.linspace(scores.min(), scores.max(), candidates):
                predictions = (scores <= threshold).astype(int)
                score = f1_score(labels, predictions)
                if score > best_f1:
                    best_threshold = float(threshold)
                    best_f1 = float(score)

            return best_threshold, best_f1
        '''
    )


def inference_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        import pandas as pd
        from sklearn.pipeline import Pipeline


        def predict_fraud_risk(
            model: Pipeline,
            features: pd.DataFrame,
            threshold: float,
        ) -> pd.DataFrame:
            """Score transactions and flag likely fraud cases."""
            scored = features.copy()
            scored["anomaly_score"] = model.decision_function(features)
            scored["is_suspicious"] = (scored["anomaly_score"] <= threshold).astype(int)
            return scored
        '''
    )


def utils_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        from pathlib import Path
        from typing import Any

        import joblib


        def save_artifact(artifact: Any, path: str | Path) -> None:
            """Persist a model or metadata artifact."""
            output_path = Path(path)
            output_path.parent.mkdir(parents=True, exist_ok=True)
            joblib.dump(artifact, output_path)


        def load_artifact(path: str | Path) -> Any:
            """Load a persisted artifact."""
            return joblib.load(Path(path))
        '''
    )

In [14]:
def streamlit_template() -> str:
    return clean_text(
        '''
        from __future__ import annotations

        import sys
        from pathlib import Path

        import pandas as pd
        import streamlit as st

        PROJECT_ROOT = Path(__file__).resolve().parents[1]
        sys.path.append(str(PROJECT_ROOT / "src"))

        from rtufd.data_generation import generate_synthetic_upi_transactions
        from rtufd.features import FEATURE_COLUMNS, build_behavioral_features
        from rtufd.inference import predict_fraud_risk
        from rtufd.modeling import train_isolation_forest, tune_threshold


        st.set_page_config(page_title="UPI Fraud Detection", layout="wide")
        st.title("Real-Time UPI Fraud Detection")

        rows = st.sidebar.slider("Synthetic transactions", 1000, 50000, 10000, step=1000)
        fraud_rate = st.sidebar.slider("Simulated fraud rate", 0.01, 0.15, 0.035, step=0.005)

        transactions = generate_synthetic_upi_transactions(rows=rows, fraud_rate=fraud_rate)
        enriched = build_behavioral_features(transactions)
        model = train_isolation_forest(enriched[FEATURE_COLUMNS])
        scores = model.decision_function(enriched[FEATURE_COLUMNS])
        threshold, f1 = tune_threshold(scores, enriched["is_fraud"])
        scored = predict_fraud_risk(model, enriched[FEATURE_COLUMNS], threshold)

        dashboard = pd.concat(
            [transactions, scored[["anomaly_score", "is_suspicious"]]],
            axis=1,
        )

        metric_columns = st.columns(3)
        metric_columns[0].metric("Transactions", f"{len(dashboard):,}")
        metric_columns[1].metric("Suspicious", f"{dashboard['is_suspicious'].sum():,}")
        metric_columns[2].metric("Best F1", f"{f1:.3f}")

        st.dataframe(dashboard.sort_values("anomaly_score").head(100), use_container_width=True)
        '''
    )


def tests_template() -> str:
    return clean_text(
        '''
        from pathlib import Path


        def test_expected_project_directories_exist() -> None:
            project_root = Path(__file__).resolve().parents[1]
            expected_directories = [
                "app",
                "config",
                "data/raw",
                "data/interim",
                "data/processed",
                "data/synthetic",
                "models",
                "notebooks",
                "reports/figures",
                "src/rtufd",
                "tests",
            ]

            missing = [path for path in expected_directories if not (project_root / path).exists()]

            assert not missing, f"Missing directories: {missing}"
        '''
    )

## 4. Create the Folder Structure

In [15]:
def create_directories(directories: Iterable[Path]) -> None:
    """Create all required project directories."""
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)


def write_file(path: Path, content: str, overwrite: bool = False) -> None:
    """Write a scaffold file while protecting existing project work by default."""
    if path.exists() and not overwrite:
        return

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def add_gitkeep_files(directories: Iterable[Path]) -> None:
    """Add .gitkeep files so empty data and artifact directories can be tracked."""
    for directory in directories:
        gitkeep_path = directory / ".gitkeep"
        if not gitkeep_path.exists():
            gitkeep_path.write_text("", encoding="utf-8")


def scaffold_project(layout: ProjectLayout, overwrite: bool = False) -> None:
    """Create directories and starter files for the UPI fraud detection project."""
    create_directories(layout.directories)
    add_gitkeep_files(layout.directories)

    for file_path, content in layout.files.items():
        write_file(file_path, content, overwrite=overwrite)


layout = ProjectLayout(root=PROJECT_ROOT, package_name=PACKAGE_NAME)
scaffold_project(layout)

print(f"Project scaffold created at: {PROJECT_ROOT}")

Project scaffold created at: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System


## 5. Display the Generated Structure

In [16]:
def show_tree(root: Path, max_depth: int = 4) -> None:
    """Print a compact project tree."""
    root = root.resolve()
    print(root.name)

    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        if depth > max_depth:
            continue

        indent = "    " * depth
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{path.name}{suffix}")


show_tree(PROJECT_ROOT)

Real-Time-UPI-Fraud-Detection-System
    .gitignore
    app/
        .gitkeep
        streamlit_app.py
    config/
        .gitkeep
        config.yaml
    data/
        interim/
            .gitkeep
        processed/
            .gitkeep
        raw/
            .gitkeep
        synthetic/
            .gitkeep
    models/
        .gitkeep
    notebooks/
        .gitkeep
    README.md
    reports/
        .gitkeep
        figures/
            .gitkeep
    requirements.txt
    src/
        rtufd/
            .gitkeep
            __init__.py
            config.py
            data_generation.py
            features.py
            inference.py
            modeling.py
            utils.py
    tests/
        .gitkeep
        test_project_structure.py


## 6. Next Commands

After running this notebook, use these commands from inside the generated project folder:

```bash
pip install -r requirements.txt
pytest
streamlit run app/streamlit_app.py
```